# Early Intervention Triage (`First Milestones`) — Property Graph, ISO GQL & Equity Audit

**Agents for Impact 2026 · Companion to `c_ei_01_load_explore.ipynb` (Section 13+)**

Run this notebook in **Colab Enterprise** right after `c_ei_01_load_explore.ipynb` (or `bash scripts/load.sh <STATE>`). It builds and verifies:

1. **`CREATE OR REPLACE PROPERTY GRAPH a4i_early_intervention.ReferralGraph`** over `CareNode` (`pediatric_and_ei_provider_directory`, excluding `provider_geom`) and `ReferralHandoff` (`deidentified_part_c_referral_logs`).
2. **Prefix-Safe Variable-Length ISO GQL (`-{2,4}->`) Breach Audit** — exposing why naive `-{2,4}->` queries double-count Type B `Vendor -> Vendor` 4-hop bounce prefixes (`34` rows vs. exact `26` statutory breaches across `44` complete episodes).
3. **Visual `ANY SHORTEST` Graph Path Widget (`TO_JSON(p)`)** — rendering multi-hop `paper_fax` bounce chains inside Colab Enterprise.
4. **Prospective `<=45`-Day Compliant Pathway Router (`GRAPH_TABLE` + `ST_DISTANCE`)** — enforcing Medicaid, Part C vendor status, bilingual capability, the **36-month Part C aging-out cliff**, and zero-vehicle mobility guardrails (with zero `audit_*` or `limited_english_rate` leakage).
5. **Counterfactual Single-Hub Fix Simulator** — quantifying delayed child-days saved by upgrading one `paper_fax` coordinator (`SYN-PROV-*-011`) to electronic intake with pre-checked bilingual/Medicaid slot availability.
6. **Post-Output Equity Audit** — joining `audit_*` demographic shares *only* after routing decisions and reporting statewide HRSA NSCH screening strata without 6x tract fan-out.

In [ ]:
import os
import pandas as pd
from google.cloud import bigquery

STATE = os.environ.get("A4I_STATE", "MS")
DATASET = os.environ.get("A4I_DATASET", "a4i_early_intervention")
LOCATION = os.environ.get("A4I_LOCATION", "US")
PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT") or bigquery.Client().project

bq = bigquery.Client(project=PROJECT, location=LOCATION)
print(f"Project : {PROJECT}")
print(f"Dataset : {PROJECT}.{DATASET} ({LOCATION})")
print(f"State   : {STATE}  [100% Synthetic CareNode & ReferralHandoff Network | Real US Census ACS Tracts]")

## 1. Declare the `ReferralGraph` Property Graph (`CareNode`, `Neighborhood`, `ReferralHandoff`, `REFERS_TO`, `SERVES_AREA`)

Notice the BigQuery Graph DDL rules enforced below:
- Unbackticked `dataset.table` inside `NODE TABLES` and `EDGE TABLES`, with unprefixed node aliases (`CareNode`, `Neighborhood`) in `REFERENCES`.
- `PROPERTIES ALL COLUMNS EXCEPT (provider_geom)` (and views excluding `tract_geom`, `limited_english_rate`, and `audit_*`) so vertex/edge properties remain scalar GQL types (`latitude` and `longitude` stay available).

In [ ]:
graph_ddl = f"""
CREATE OR REPLACE VIEW {DATASET}.v_graph_neighborhoods AS
SELECT
  LPAD(census_tract_geoid, 11, '0') AS tract_geoid,
  state_code,
  county_name,
  children_under_5_count,
  poverty_rate,
  no_vehicle_rate,
  assistance_rate,
  median_household_income,
  latitude,
  longitude
FROM {DATASET}.acs_under5_tract_demographics;

CREATE OR REPLACE VIEW {DATASET}.v_graph_refers_to_edges AS
SELECT
  referral_event_id,
  source_provider_id,
  target_provider_id,
  case_chain_id,
  hop_sequence,
  handoff_wait_days,
  transmission_channel,
  handoff_reason,
  concern_domain,
  child_age_months,
  child_home_tract_geoid,
  IF(STARTS_WITH(handoff_reason, 'Re-Referral Bounce'), 1.0, 0.0) AS bounce_flag
FROM {DATASET}.deidentified_part_c_referral_logs;

CREATE OR REPLACE VIEW {DATASET}.v_graph_serves_area_edges AS
SELECT
  CONCAT(p.provider_id, '_', LPAD(n.census_tract_geoid, 11, '0')) AS serves_edge_id,
  p.provider_id,
  LPAD(n.census_tract_geoid, 11, '0') AS tract_geoid,
  ROUND(ST_DISTANCE(ST_GEOGPOINT(p.longitude, p.latitude), ST_GEOGPOINT(n.longitude, n.latitude)) / 1000.0, 2) AS travel_distance_km,
  CAST(ROUND((ST_DISTANCE(ST_GEOGPOINT(p.longitude, p.latitude), ST_GEOGPOINT(n.longitude, n.latitude)) / 1000.0) * 1.5, 0) AS INT64) AS estimated_travel_minutes,
  (p.node_role IN ('Part_C_Intake_Coordinator', 'Multidisciplinary_Eval_Hub') AND p.bilingual_services_listed) AS mobile_visit_capable
FROM {DATASET}.pediatric_and_ei_provider_directory AS p
JOIN {DATASET}.acs_under5_tract_demographics AS n
  ON p.state_code = n.state_code
WHERE ST_DISTANCE(ST_GEOGPOINT(p.longitude, p.latitude), ST_GEOGPOINT(n.longitude, n.latitude)) <= 80000;

CREATE OR REPLACE PROPERTY GRAPH {DATASET}.ReferralGraph
  NODE TABLES (
    {DATASET}.pediatric_and_ei_provider_directory AS CareNode
      KEY (provider_id)
      LABEL CareNode
      PROPERTIES ALL COLUMNS EXCEPT (provider_geom),
    {DATASET}.v_graph_neighborhoods AS Neighborhood
      KEY (tract_geoid)
      LABEL Neighborhood
      PROPERTIES ALL COLUMNS
  )
  EDGE TABLES (
    {DATASET}.deidentified_part_c_referral_logs AS ReferralHandoff
      KEY (referral_event_id)
      SOURCE KEY (source_provider_id) REFERENCES CareNode (provider_id)
      DESTINATION KEY (target_provider_id) REFERENCES CareNode (provider_id)
      LABEL ReferralHandoff
      PROPERTIES ALL COLUMNS,
    {DATASET}.v_graph_refers_to_edges AS REFERS_TO
      KEY (referral_event_id)
      SOURCE KEY (source_provider_id) REFERENCES CareNode (provider_id)
      DESTINATION KEY (target_provider_id) REFERENCES CareNode (provider_id)
      LABEL REFERS_TO
      PROPERTIES ALL COLUMNS,
    {DATASET}.v_graph_serves_area_edges AS SERVES_AREA
      KEY (serves_edge_id)
      SOURCE KEY (provider_id) REFERENCES CareNode (provider_id)
      DESTINATION KEY (tract_geoid) REFERENCES Neighborhood (tract_geoid)
      LABEL SERVES_AREA
      PROPERTIES ALL COLUMNS
  );
"""
bq.query(graph_ddl).result()
print(f"Declared {PROJECT}.{DATASET}.ReferralGraph successfully.")

## 2. Prefix-Safe Variable-Length `-{2,4}->` ISO GQL Breach Audit

In every state, 8 of the 26 breached referral chains follow a **Type B 4-hop Vendor Bounce** (`Clinic -> Coordinator -> Eval Hub -> Capped EI_Therapy_Vendor (025) -> Fast EI_Therapy_Vendor (026..030)`). Because both Hop 3 and Hop 4 land on `node_role = 'EI_Therapy_Vendor'`, a naive `-{2,4}->` query matches **both** the 3-hop prefix and the 4-hop complete path!

Below we enforce:
1. `hops[OFFSET(0)].hop_sequence = 1` (starts at true episode origin)
2. `(SELECT COUNT(DISTINCT h.case_chain_id) FROM UNNEST(hops) AS h) = 1` (correlates `case_chain_id` across every hop)
3. `QUALIFY ROW_NUMBER() OVER (PARTITION BY chain_id ORDER BY total_hops DESC) = 1` (eliminates 3-hop prefix duplicates on 4-hop `Vendor -> Vendor` chains)

In [ ]:
breach_gql = f"""
WITH complete_chains AS (
  SELECT *
  FROM GRAPH_TABLE(
    {DATASET}.ReferralGraph
    MATCH (clinic:CareNode {{node_role: 'Screening_Pediatric_Clinic'}})
          -[hops:ReferralHandoff]->{{2,4}}
          (vendor:CareNode {{node_role: 'EI_Therapy_Vendor'}})
    WHERE hops[OFFSET(0)].hop_sequence = 1
      AND (SELECT COUNT(DISTINCT h.case_chain_id) FROM UNNEST(hops) AS h) = 1
    LET chain_id = hops[OFFSET(0)].case_chain_id,
        total_hops = ARRAY_LENGTH(hops),
        child_age_months = hops[OFFSET(0)].child_age_months,
        concern_domain = hops[OFFSET(0)].concern_domain,
        home_tract = hops[OFFSET(0)].child_home_tract_geoid,
        intake_coord_id = hops[OFFSET(0)].target_provider_id,
        chain_wait_days = (SELECT SUM(h.handoff_wait_days) FROM UNNEST(hops) AS h),
        fax_hops = (SELECT COUNTIF(h.transmission_channel = 'paper_fax') FROM UNNEST(hops) AS h),
        bounce_reason = (
          SELECT ANY_VALUE(h.handoff_reason)
          FROM UNNEST(hops) AS h
          WHERE STARTS_WITH(h.handoff_reason, 'Re-Referral Bounce')
        ),
        traced_route = (
          SELECT STRING_AGG(
            CONCAT(h.source_provider_id, ' -[', h.transmission_channel, ' ', CAST(h.handoff_wait_days AS STRING), 'd]-> '),
            '' ORDER BY h.hop_sequence
          )
          FROM UNNEST(hops) AS h
        )
    RETURN
      chain_id,
      child_age_months,
      concern_domain,
      home_tract,
      intake_coord_id,
      vendor.provider_id AS final_vendor_id,
      total_hops,
      fax_hops,
      chain_wait_days,
      bounce_reason,
      traced_route
  )
  QUALIFY ROW_NUMBER() OVER (PARTITION BY chain_id ORDER BY total_hops DESC) = 1
)
SELECT
  chain_id,
  child_age_months,
  ROUND(child_age_months + chain_wait_days / 30.44, 1) AS est_age_at_ifsp_m,
  concern_domain,
  intake_coord_id,
  total_hops,
  fax_hops,
  chain_wait_days,
  bounce_reason,
  CONCAT(traced_route, final_vendor_id) AS full_path
FROM complete_chains
WHERE chain_wait_days > 45
ORDER BY child_age_months DESC, chain_wait_days DESC
LIMIT 12;
"""
display(bq.query(breach_gql).to_dataframe())

## 3. Colab Graph Visualization (`TO_JSON(p)`)

Returning `TO_JSON(p)` over `case_chain_id`-correlated `-{3,4}->` trajectories starting at `hop_sequence = 1` activates BigQuery/Colab's native interactive graph renderer while staying well below the 2 MB payload limit.

In [ ]:
viz_gql = f"""
GRAPH {DATASET}.ReferralGraph
MATCH p =
  (clinic:CareNode {{node_role: 'Screening_Pediatric_Clinic'}})
  -[hops:ReferralHandoff]->{{3,4}}
  (vendor:CareNode {{node_role: 'EI_Therapy_Vendor'}})
WHERE hops[OFFSET(0)].hop_sequence = 1
  AND (SELECT COUNT(DISTINCT h.case_chain_id) FROM UNNEST(hops) AS h) = 1
  AND (SELECT SUM(h.handoff_wait_days) FROM UNNEST(hops) AS h) > 45
RETURN TO_JSON(p) AS breach_paths
LIMIT 15;
"""
display(bq.query(viz_gql).to_dataframe())

## 4. Prospective `<=45`-Day Compliant Pathway Router (`GRAPH_TABLE` + `ST_DISTANCE`)

Finds Medicaid-enrolled, bilingual `Clinic -> Coordinator -> Eval Hub -> Therapy Vendor` routes clearing the 45-day IDEA Part C ceiling (`34 CFR § 303.310`) for the highest-poverty under-5 tract, computing exact `ST_DISTANCE` from `ST_CENTROID(tract_geom)` to flag zero-vehicle NEMT / mobile evaluation needs (strictly excluding `audit_*` and all-zero `limited_english_rate`).

In [ ]:
routing_sql = f"""
WITH target_tract AS (
  SELECT
    LPAD(census_tract_geoid, 11, '0') AS tract_geoid,
    county_name,
    poverty_rate,
    no_vehicle_rate,
    tract_geom
  FROM `{PROJECT}.{DATASET}.acs_under5_tract_demographics`
  ORDER BY poverty_rate DESC, children_under_5_count DESC
  LIMIT 1
),
graph_candidates AS (
  SELECT *
  FROM GRAPH_TABLE(
    {DATASET}.ReferralGraph
    MATCH (n:Neighborhood)
          <-[s:SERVES_AREA]-
          (clinic:CareNode {{node_role: 'Screening_Pediatric_Clinic', accepts_medicaid: TRUE}})
          -[h1:REFERS_TO]->
          (coord:CareNode {{node_role: 'Part_C_Intake_Coordinator', part_c_vendor_enrolled: TRUE, accepts_medicaid: TRUE, bilingual_services_listed: TRUE}})
          -[h2:REFERS_TO]->
          (eval:CareNode {{node_role: 'Multidisciplinary_Eval_Hub', part_c_vendor_enrolled: TRUE, accepts_medicaid: TRUE, bilingual_services_listed: TRUE}})
          -[h3:REFERS_TO]->
          (vendor:CareNode {{node_role: 'EI_Therapy_Vendor', part_c_vendor_enrolled: TRUE, accepts_medicaid: TRUE, bilingual_services_listed: TRUE}})
    WHERE h1.bounce_flag = 0.0 AND h2.bounce_flag = 0.0 AND h3.bounce_flag = 0.0
    LET expected_days = coord.median_internal_wait_days + eval.median_internal_wait_days + vendor.median_internal_wait_days
    FILTER expected_days <= 45
    RETURN
      n.tract_geoid AS served_tract,
      clinic.provider_id AS clinic_id,
      coord.provider_id AS coord_id,
      eval.provider_id AS eval_id,
      vendor.provider_id AS vendor_id,
      expected_days,
      eval.latitude AS eval_lat,
      eval.longitude AS eval_lon,
      vendor.latitude AS vendor_lat,
      vendor.longitude AS vendor_lon
  )
)
SELECT DISTINCT
  t.tract_geoid,
  t.county_name,
  ROUND(t.poverty_rate * 100, 1) AS poverty_pct,
  ROUND(t.no_vehicle_rate * 100, 1) AS no_vehicle_pct,
  g.clinic_id,
  g.coord_id,
  g.eval_id,
  g.vendor_id,
  g.expected_days,
  ROUND(ST_DISTANCE(ST_CENTROID(t.tract_geom), ST_GEOGPOINT(g.eval_lon, g.eval_lat)) / 1000.0, 1) AS eval_km,
  ROUND(ST_DISTANCE(ST_CENTROID(t.tract_geom), ST_GEOGPOINT(g.vendor_lon, g.vendor_lat)) / 1000.0, 1) AS vendor_km,
  (t.no_vehicle_rate >= 0.08 AND ST_DISTANCE(ST_CENTROID(t.tract_geom), ST_GEOGPOINT(g.eval_lon, g.eval_lat)) > 25000) AS requires_nemt_or_mobile_eval
FROM graph_candidates AS g
JOIN target_tract AS t
  ON g.served_tract = t.tract_geoid
ORDER BY g.expected_days ASC, eval_km ASC
LIMIT 8;
"""
display(bq.query(routing_sql).to_dataframe())

## 5. Counterfactual Single-Hub Intervention Simulator

What happens if we intervene at the top hidden `Part_C_Intake_Coordinator` bounce hub (`SYN-PROV-*-011`, whose flat `median_internal_wait_days` is just `5` days!) by:
1. Converting outbound `paper_fax` handoffs (`~21.7d` avg) to `ehr_electronic_portal` (`~13.0d` avg), and
2. Pre-verifying bilingual & Medicaid evaluation slot capacity at intake so toddlers route directly into same-city fast evaluation hubs (`017..020`) and fast vendors (`026..030`) instead of bouncing off saturated hubs (`013..016` / `025`)?

In [ ]:
counterfactual_sql = f"""
WITH chains AS (
  SELECT *
  FROM GRAPH_TABLE(
    {DATASET}.ReferralGraph
    MATCH (clinic:CareNode {{node_role: 'Screening_Pediatric_Clinic'}})
          -[hops:ReferralHandoff]->{{2,4}}
          (vendor:CareNode {{node_role: 'EI_Therapy_Vendor'}})
    WHERE hops[OFFSET(0)].hop_sequence = 1
      AND (SELECT COUNT(DISTINCT h.case_chain_id) FROM UNNEST(hops) AS h) = 1
    LET chain_id = hops[OFFSET(0)].case_chain_id,
        total_hops = ARRAY_LENGTH(hops),
        coord_id = hops[OFFSET(0)].target_provider_id,
        baseline_wait_d = (SELECT SUM(h.handoff_wait_days) FROM UNNEST(hops) AS h),
        fax_savings_d = (
          SELECT IFNULL(SUM(GREATEST(h.handoff_wait_days - 12, 0)), 0)
          FROM UNNEST(hops) AS h
          WHERE h.transmission_channel = 'paper_fax'
            AND NOT STARTS_WITH(h.handoff_reason, 'Re-Referral Bounce')
        ),
        bounce_hop_d = (
          SELECT IFNULL(SUM(h.handoff_wait_days), 0)
          FROM UNNEST(hops) AS h
          WHERE STARTS_WITH(h.handoff_reason, 'Re-Referral Bounce')
        )
    RETURN chain_id, total_hops, coord_id, baseline_wait_d, fax_savings_d, bounce_hop_d
  )
  QUALIFY ROW_NUMBER() OVER (PARTITION BY chain_id ORDER BY total_hops DESC) = 1
)
SELECT
  c.coord_id,
  p.synthetic_organization_name,
  p.median_internal_wait_days AS flat_internal_wait_d,
  COUNT(*) AS total_routed_cases,
  COUNTIF(c.baseline_wait_d > 45) AS baseline_breaches,
  COUNTIF((c.baseline_wait_d - c.fax_savings_d - c.bounce_hop_d) > 45) AS post_fix_breaches,
  SUM(c.fax_savings_d + c.bounce_hop_d) AS total_child_days_saved
FROM chains AS c
JOIN `{PROJECT}.{DATASET}.pediatric_and_ei_provider_directory` AS p
  ON c.coord_id = p.provider_id
GROUP BY 1, 2, 3
ORDER BY total_child_days_saved DESC;
"""
display(bq.query(counterfactual_sql).to_dataframe())

## 6. Post-Output Equity Audit & Statewide NSCH Context

**Strict Governance Rule:** `audit_*` demographic shares are never included in routing queries or priority weights. Below we join `audit_*`, `poverty_rate`, and `no_vehicle_rate` strictly *after* generating pathway recommendations, and inspect statewide HRSA NSCH language strata separately (`6` statewide rows, zero tract fan-out).

In [ ]:
equity_sql = f"""
WITH episodes AS (
  SELECT
    case_chain_id,
    ANY_VALUE(child_home_tract_geoid) AS child_home_tract_geoid,
    SUM(handoff_wait_days) AS episode_wait_days
  FROM `{PROJECT}.{DATASET}.deidentified_part_c_referral_logs`
  GROUP BY case_chain_id
)
SELECT
  IF(e.episode_wait_days > 45, 'Breached (>45d)', 'Timely (<=45d)') AS statutory_status,
  COUNT(*) AS child_episodes,
  ROUND(AVG(e.episode_wait_days), 1) AS mean_wait_days,
  ROUND(AVG(d.poverty_rate) * 100, 1) AS tract_poverty_pct,
  ROUND(AVG(d.no_vehicle_rate) * 100, 1) AS zero_vehicle_pct,
  ROUND(AVG(d.audit_black_share) * 100, 1) AS audit_black_pct,
  ROUND(AVG(d.audit_hispanic_share) * 100, 1) AS audit_hispanic_pct,
  ROUND(AVG(d.audit_indigenous_share) * 100, 1) AS audit_indigenous_pct,
  ROUND(AVG(d.audit_asian_share) * 100, 1) AS audit_asian_pct,
  ROUND(AVG(d.audit_white_share) * 100, 1) AS audit_white_pct
FROM episodes AS e
JOIN `{PROJECT}.{DATASET}.acs_under5_tract_demographics` AS d
  ON e.child_home_tract_geoid = d.census_tract_geoid
GROUP BY 1
ORDER BY 1;
"""
print("1) Post-Output Neighborhood Equity Audit (ACS Real Tracts joined to Synthetic Episode Outcomes):")
display(bq.query(equity_sql).to_dataframe())

nsch_sql = f"""
SELECT
  survey_stratum_id,
  geographic_level,
  household_income_fpl_band,
  primary_home_language,
  ROUND(standardized_screening_rate_pct * 100, 1) AS screening_rate_pct,
  ROUND(followup_requested_pct * 100, 1) AS followup_requested_pct
FROM `{PROJECT}.{DATASET}.nsch_developmental_screening_survey`
WHERE geographic_level = 'Statewide'
ORDER BY screening_rate_pct ASC;
"""
print("\n2) Statewide HRSA NSCH Developmental Screening Strata (6 Statewide rows — never raw-joined to tracts!):")
display(bq.query(nsch_sql).to_dataframe())